In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Torch codec fix
import os, sys
npp_path = f"{sys.prefix}/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages/nvidia/cu13/lib"
os.environ["LD_LIBRARY_PATH"] = f"{npp_path}:{os.environ.get('LD_LIBRARY_PATH', '')}"
print("LD_LIBRARY_PATH set to:", os.environ["LD_LIBRARY_PATH"])

import ctypes, sys
libdir = f"{sys.prefix}/lib/python{sys.version_info.major}.{sys.version_info.minor}/site-packages/nvidia/cu13/lib"
for so in ("libnppc.so.13", "libnppig.so.13", "libnppicc.so.13"):
    ctypes.CDLL(f"{libdir}/{so}", mode=ctypes.RTLD_GLOBAL)
print("Preloaded NPP libs from:", libdir)

LD_LIBRARY_PATH set to: /home/ubuntu/diffusion/venv/lib/python3.10/site-packages/nvidia/cu13/lib:/usr/mpi/gcc/openmpi-4.1.7rc1/lib:/usr/mpi/gcc/openmpi-4.1.7rc1/lib64
Preloaded NPP libs from: /home/ubuntu/diffusion/venv/lib/python3.10/site-packages/nvidia/cu13/lib


In [3]:
import os
import sys
sys.path.insert(0, "..")

from dataclasses import dataclass

import pandas as pd
from torchvision import transforms

import torch.nn as nn
from torch.utils.data.dataset import Dataset
from torch.utils.data.dataloader import DataLoader

from src.utils import show_sequence_sample, show_sequence_gif
from src.dataset import CSDataset

from tqdm.auto import tqdm
from torchvision import transforms

from src.utils import save_ckpt, load_ckpt
from IPython.display import Image as IPyImage

/home/ubuntu/diffusion/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset
from PIL import Image
import numpy as np
from torchvision import transforms as T

BUTTONS = [
    "FORWARD", "LEFT", "RIGHT", "BACK",
    "FIRE", "RIGHTCLICK", "RELOAD", "INSPECT",
]

In [5]:
# Use the VAE to encode the images, then train our image diffusion (flow matching) model on this
DATA_PATH = "/home/ubuntu/diffusion/data"

video_dataset = CSDataset(DATA_PATH, T=30, stride=2) #sampling 30fps from original 60fps videos
print(f"Dataset length: {len(video_dataset)}")


sample = video_dataset[2]
sample_frames = sample['video'].unsqueeze(0)
print(sample_frames.shape) #15, 3, 224, 224

#Visualize matplotlib
# show_sequence_sample({"video": sample_frames}, num_timesteps=30)

Dataset length: 109993
torch.Size([1, 30, 3, 224, 224])


In [6]:
# #Verify data loader
# batch_size = 4
# data_loader = DataLoader(video_dataset, batch_size, shuffle=True, num_workers=4, pin_memory=True, prefetch_factor=2) #For pre-processing must ensure order is correct thus shuffle=False
# batch = next(iter(data_loader))
# print(f"Sequence shape: {batch['video'].shape}")

# #This looks roughly correct. clips starting at frames 0, 1, 2 and 3
# for i in range(batch_size):
#     show_sequence_sample(batch=batch, num_timesteps=15, batch_idx=i)

In [7]:
import lpips
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'

lpips_fn = lpips.LPIPS(net="vgg").eval().to(device)
for p in lpips_fn.parameters():
    p.requires_grad_(False)

def perceptual_lpips(x_hat, x):
    # LPIPS expects [-1,1] float tensors, NCHW
    return lpips_fn(x_hat, x).mean()

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/ubuntu/diffusion/venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/ubuntu/diffusion/venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/ubuntu/diffusion/venv/lib/python3.10/site-packages/lpips/weights/v0.1/vgg.pth


In [8]:
# Video is highly-compressible (ex: codecs have ~200-2000x compression)
# Thus, we'd expect significant compression to train a VAE sequences rather than individual frames
# Additionally - we can avoid e.g. frame subsampling, framerate reduction (which reduces FPS 'slow motion')

# Note:
# Image size: 224 * 224 * 3 = 150,528
# Seq length: original is 30fps? 
# Bottleneck size: 4 * 28 * 28 = 3,136
# Compression ~x48 (this matches up with Stable Diffusion compression)

@dataclass
class VideoVAEConfig:
    beta: float = 1e-6
    lam_perc: float = 0.1
    T: int = 30

class VideoVAE(nn.Module):

    def __init__(self, config: VideoVAEConfig) -> None:
        super().__init__()
        self.config = config

        #KL Term weight. Too high beta leads to mean/over-regularization
        self.beta = config.beta
        self.lam_perc = config.lam_perc
        self.T = config.T

        # (B, 3, T, 224, 224) -> (B, 32, T, 112, 112)
        self.enc1 = nn.Sequential(nn.Conv3d(3, 32, kernel_size=3, stride=(1, 2, 2), padding=1), nn.BatchNorm3d(32), nn.ReLU())

        # (B, 32, 112, 112) -> (B, 64, 56, 56)
        self.enc2 = nn.Sequential(nn.Conv3d(32, 64, kernel_size=3, stride=(1, 2, 2), padding=1), nn.BatchNorm3d(64), nn.ReLU())

        # (B, 64, 56, 56) -> (B, 128, 28, 28) (note: visualize the latents to understand what the model is learning)
        self.enc3 = nn.Sequential(nn.Conv3d(64, 128, kernel_size=3, stride=(1, 2, 2), padding=1), nn.BatchNorm3d(128), nn.ReLU())

        self.enc4 = nn.Sequential(nn.Conv3d(128, 4, kernel_size=1))

        #Variational autoencoder -> we predict mu and log(var). kernel_size 1 as we predicting mean and logvar per latent dim
        self.mu_head = nn.Conv3d(4, 4, kernel_size=1)
        self.logvar_head = nn.Conv3d(4, 4, kernel_size=1)
        
        self.dec1 = nn.Sequential(nn.ConvTranspose3d(4, 64, kernel_size=3, stride=(1, 2, 2), padding=1, output_padding=(0,1,1)), nn.BatchNorm3d(64),nn.ReLU())

        # Note: Here we remove the skip connections - VAE are generative, thus if we sample from noise, it won't know the input x for skip connections
        self.dec2 = nn.Sequential(nn.ConvTranspose3d(64, 32, kernel_size=3, stride=(1, 2, 2), padding=1, output_padding=(0,1,1)), nn.BatchNorm3d(32), nn.ReLU())

        self.dec3 = nn.Sequential(nn.ConvTranspose3d(32, 3, kernel_size=3, stride=(1, 2, 2), padding=1, output_padding=(0,1,1)), nn.Tanh())

    def encode(self, x):
        h = self.enc4(self.enc3(self.enc2(self.enc1(x)))) #hidden layer output
        mu = self.mu_head(h) #predict the means for each latent dim channel, given h
        logvar = self.logvar_head(h) #predict the logvars for each latent dim channel, given h
        return mu, logvar

    def decode(self, z):
        return self.dec3(self.dec2(self.dec1(z)))

    def reparametrize(self, mu, logvar):
        #this allows us to backpropagate through mean and std (yet sample noise)

        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std) #sample gaussian noise.

        return mu + eps * std #this trick allows us to backprop the gradients to mu and std, while sampling from gaussian noise

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.permute(0, 2, 1, 3, 4).contiguous() #Conv3D expects channel as 2nd dimension

        mu, logvar = self.encode(x)
        z = self.reparametrize(mu, logvar)
        x_hat = self.decode(z) #i.e. learn to decode z from a noisy distribution (not from the direct latent). This means there's no gaps/i.e. smooth interp for each latent dim

        recon = nn.functional.l1_loss(x_hat, x, reduction="mean") #Ensure pixel-reconstruction is similar. l1 loss give sharper images thanMSE. (MSE incentivizes reducing large errors -> averaging out uncertain details)

        x = x.permute(0, 2, 1, 3, 4)
        x_hat = x_hat.permute(0, 2, 1, 3, 4)
        
        #perceptual loss is on 
        # x_frames = x.reshape(B*T, C, H, W)
        # x_hat_frames = x_hat.reshape(B*T, C, H, W)

        # # perc = perceptual_lpips(x_hat_frames, x_frames) #TODO: Too slow, call only on first frame
        # perc = perceptual_lpips(x_hat_frames[0], x_frames[0])

        #KL divergence -> regularize the prediction so that mean and std are roughly gaussian (ensures no drastically changing latents)
        kl_per_elem = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp()) #kl is simple for 2 gaussian distributions
        kl = kl_per_elem.sum(dim=(1,2,3,4)).mean() #mean over the batch
        
        #reconstruction loss pushes model to reconstruct images
        #kl loss is a regularizer, pushes latent space to be roughly gaussian (leads to blurry reconstructions)
        #perceptual loss is important - leads to far crisper images
        #perhaps need Patch-GAN loss (used by Stable Diffusion) for further crisp images
        # loss = recon + self.lam_perc * perc + self.beta * kl
        loss = recon + self.beta * kl

        # return loss, x_hat, recon.detach(), kl.detach(), perc.detach()
        return loss, x_hat, recon.detach(), kl.detach()

    @torch.no_grad()
    def sample(self, batch_size, device, shape=None):
        if not shape:
            shape = (4, self.T, 28, 28)
        z = torch.randn(batch_size, *shape, device=device)
        return self.decode(z)


In [ ]:

# @dataclass
# class TrainConfig:
#     epochs: int = 1
#     batch_size: int = 32
#     lr: float = 1e-3
#     T: int = 10 #at 30fps, 0.5s of video

# device = 'cuda' if torch.cuda.is_available() else 'cpu'

# train_config = TrainConfig()

# #Variational Autoencoder
# model_config = VideoVAEConfig(T=train_config.T)
# model = VideoVAE(model_config).to(device)

# optimizer = torch.optim.AdamW(params=model.parameters(), lr=train_config.lr)

# video_dataset = CSDataset(DATA_PATH, T=train_config.T, stride=3)
# train_data_loader = DataLoader(video_dataset, train_config.batch_size, shuffle=True)

# losses = []
# recons = []
# kls = []
# percs = []

# model.train() #ensure to set train mode (batch norm requires this for computing means/norms)

# global_step = 0

# for epoch in range(train_config.epochs):
#     pbar = tqdm(train_data_loader, desc=f"Epoch {epoch + 1}/{train_config.epochs}", leave=True)

#     for batch in pbar:
#         frames = batch['video'].to(device)
#         actions = batch['actions'].to(device)

#         #Make sure to zero grad (otherwise grad accumulates across epochs)
#         optimizer.zero_grad()

#         #compute the forward pass
#         loss, _, recon, kl, perc = model(frames)

#         #compute the partial derivatives
#         loss.backward()

#         #update the params: wi1 = wi0 - dL/dwi
#         optimizer.step()

#         pbar.set_postfix(
#             loss=f"{loss.item():.4f}",
#             recon=f"{recon.item():.4f}",
#             bkl=f"{(model_config.beta * kl.item()):.4f}",
#             perc=f"{perc.item():.4f}",
#         )

#     if global_step % 100 == 0:    
#         losses.append(loss.item())
#         recons.append(recon.item())
#         kls.append(model_config.beta * kl.item())
#         percs.append(perc.item())

# model.eval()

: 

In [ ]:
from dataclasses import dataclass
from itertools import cycle

@dataclass
class TrainConfig:
    max_steps: int = 50000
    batch_size: int = 128
    lr: float = 1e-3
    T: int = 10
    log_every: int = 100
    sample_every: int = 20000

device = "cuda" if torch.cuda.is_available() else "cpu"

train_config = TrainConfig()

model_config = VideoVAEConfig(T=train_config.T)
model = VideoVAE(model_config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=train_config.lr)

video_dataset = CSDataset(DATA_PATH, T=train_config.T, stride=3)
train_data_loader = DataLoader(
    video_dataset,
    batch_size=train_config.batch_size,
    shuffle=True,
    num_workers=32,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=2
)

losses, recons, kls, percs = [], [], [], []

model.train()
global_step = 0
data_iter = cycle(train_data_loader)

pbar = tqdm(total=train_config.max_steps, desc="Training", leave=True)

while global_step < train_config.max_steps:
    batch = next(data_iter)

    frames = batch["video"].to(device, non_blocking=True)
    # actions = batch["actions"].to(device)

    optimizer.zero_grad(set_to_none=True)

    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        # loss, x_hat, recon, kl, perc = model(frames)
        loss, x_hat, recon, kl = model(frames)
    
    loss.backward()
    optimizer.step()

    global_step += 1
    pbar.update(1)

    pbar.set_postfix(
        step=global_step,
        loss=f"{loss.item():.4f}",
        recon=f"{recon.item():.4f}",
        bkl=f"{(model_config.beta * kl.item()):.4f}",
        # perc=f"{perc.item():.4f}",
    )

    if global_step % train_config.log_every == 0:
        losses.append(loss.item())
        recons.append(recon.item())
        kls.append(model_config.beta * kl.item())
        # percs.append(perc.item())

    if global_step % train_config.sample_every == 0:
        model.eval()
        with torch.no_grad():
            preview = x_hat[:4].detach().cpu()
        model.train()

        print(f"Step {global_step}: loss={loss.item():.4f}")
        # visualize preview here
        # e.g. show first frame from each clip

pbar.close()
model.eval()

Training:   1%|          | 386/50000 [16:31<55:12:20,  4.01s/it, bkl=0.0193, loss=0.1249, recon=0.1056, step=386] 

In [ ]:
CHECKPOINTS_DIR = f"/home/ubuntu/diffusion/checkpoints/video_vae/vae_{train_config.max_steps}_epochs_draft.pt"
save_ckpt(CHECKPOINTS_DIR, model, optimizer, epoch=train_config.max_steps)

In [ ]:
import matplotlib.pyplot as plt

#plot the losses
fig, axes = plt.subplots(1,4, figsize=(12, 4))
axes[0].plot(losses)
axes[1].plot(recons)
axes[2].plot(kls)
# axes[3].plot(percs)

plt.show()

print("Loss:", losses[-1])
print("Recon:", recons[-1])
print("KL:", kls[-1])
# print("Perc", percs[-1])

In [ ]:
# Visualize the trained video vae
model.eval()

with torch.no_grad():
    batch  = next(iter(train_data_loader))
    sequence = batch['video'].to(device)
    
    with torch.no_grad():
        _, output, _, _ = model(sequence)
    
    output_sequence = output.cpu()

    print(f"Reconstruction loss: {loss.item():.4f}")
    print("Original:")
    show_sequence_sample(batch, num_timesteps=15, batch_idx=0)

    # show_batch_sample(batch, 5)
    print("Reconstructed:")
    show_sequence_sample({"video": output_sequence}, num_timesteps=15, batch_idx=0)

In [ ]:
CHECKPOINT = f"/home/ubuntu/diffusion/checkpoints/video_vae/vae_{train_config.max_steps}_epochs_draft.pt"

vae_config = VideoVAEConfig()
model = VideoVAE(vae_config)

load_ckpt(CHECKPOINT, model, map_location='cuda')
model.to(device)
model.eval()

In [ ]:
#Verify loaded model works
# Visualize the trained video vae

with torch.no_grad():
    batch  = next(iter(train_data_loader))
    sequence = batch['video'].to(device)
    
    with torch.no_grad():
        _, output, _, _ = model(sequence)
    
    output_sequence = output.cpu()
    output_batch = {"video": output_sequence}

    print("Original (gif):")
    output_path = "/home/ubuntu/diffusion/tmp"
    gif_path = show_sequence_gif(batch, num_timesteps=train_config.T, filename='original.gif', output_path=output_path, batch_idx=0)
    display(IPyImage(filename=gif_path))

    print("Reconstructed (gif):")
    recon_gif_path = show_sequence_gif(output_batch, num_timesteps=train_config.T, filename='reconstructed.gif', output_path=output_path, batch_idx=0)
    display(IPyImage(filename=recon_gif_path))

    print(f"Reconstruction loss: {loss.item():.4f}")
    print("Original:")
    show_sequence_sample(batch, num_timesteps=5, batch_idx=0)
    
    print("Reconstructed:")
    show_sequence_sample(output_batch, num_timesteps=5, batch_idx=0)

In [ ]:
# Profiling
import time, torch

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

# 1) dataloader only
t0 = time.perf_counter()
for _ in range(20):
    batch = next(data_iter)
t1 = time.perf_counter()
print("dataloader only:", (t1 - t0) / 20)

# 2) model only, synthetic data
frames = torch.randn(32, 10, 3, 224, 224, device=device)
sync()
t0 = time.perf_counter()
for _ in range(10):
    optimizer.zero_grad(set_to_none=True)
    loss, *_ = model(frames)
    loss.backward()
    optimizer.step()
sync()
t1 = time.perf_counter()
print("model+backward only:", (t1 - t0) / 10)

# 3) memory peak
torch.cuda.reset_peak_memory_stats()
optimizer.zero_grad(set_to_none=True)
loss, *_ = model(frames)
loss.backward()
sync()
print("peak GB:", torch.cuda.max_memory_allocated() / 1024**3)